In [1]:
''' This does all the prep for making a BlendedMLP: it splits a mesh into patches
using geodesics, and parametrises the patches using Tutte's embedding. '''

import trimesh
import open3d as o3d
import numpy as np

from scipy.spatial import cKDTree
from mesh_processing import *
import matplotlib.pyplot as plt

import os
import shutil
from visuals import *

original_cwd = os.getcwd()

In [2]:
# Settings
shape_name =  'SimpleHuman'
manual_decimation = True

target_number_of_triangles = 25

#bear in mind 'infinity' is quicker than a big subdivision level. Only use subdivision for small numbers.
subdivision_level = 0 #'infinity' # 0,1,2, ... or 'infinity'



In [3]:
''' Load mesh (fine_mesh) '''


# Paths

off_filepath = f"data/surfaces/{shape_name}.off"
obj_filepath = f"data/surfaces/{shape_name}.obj"
manual_coarse_filepath = f"data/surfaces/{shape_name}-coarse-m.obj"

# Check if .off file exists
if not os.path.exists(off_filepath):
    if os.path.exists(obj_filepath):
        print(f"{off_filepath} not found. Converting from .obj...")
        mesh = o3d.io.read_triangle_mesh(obj_filepath)
    else:
        raise FileNotFoundError(f"Neither {off_filepath} nor {obj_filepath} found.")
else:
    mesh = o3d.io.read_triangle_mesh(off_filepath)

# Normalize the mesh
mesh.vertex_normals = o3d.utility.Vector3dVector()  # Clear vertex normals
mesh.triangle_normals = o3d.utility.Vector3dVector()  # Clear triangle normals
mesh = normalise_to_unit_bbox(mesh)

# Overwrite both .off and .obj with normalized mesh
o3d.io.write_triangle_mesh(off_filepath, mesh)
o3d.io.write_triangle_mesh(obj_filepath, mesh)

print(f"Normalized mesh saved to {off_filepath} and {obj_filepath}.")






1.0
[Open3D WARNING] Write OFF cannot include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
Normalized mesh saved to data/surfaces/SimpleHuman.off and data/surfaces/SimpleHuman.obj.


In [6]:
''' simplified mesh (coarse_mesh) '''

# Load mesh from .off
fine_mesh = o3d.io.read_triangle_mesh(off_filepath)


if manual_decimation == False:

    # Simplify mesh
    simplified_mesh = fine_mesh.simplify_quadric_decimation(
        target_number_of_triangles=target_number_of_triangles
    )

else:
    simplified_mesh = o3d.io.read_triangle_mesh(manual_coarse_filepath)
    # Voxel downsample the vertices to merge close points
    #voxel_size = 0.001  # Adjust based on your mesh scale
    #simplfied_mesh = mesh.simplify_vertex_clustering(voxel_size)
    
    # Remove any duplicated vertices if they still exist
    simplified_mesh.remove_duplicated_vertices()

    # Ensure consistent face orientation
    simplified_mesh.orient_triangles()
    
    
    # Remove degenerate triangles
    simplified_mesh.remove_degenerate_triangles()
    
    # Remove non-manifold edges
    simplified_mesh.remove_non_manifold_edges()
    
    # Optionally remove isolated vertices (if some remain)
    simplified_mesh.remove_unreferenced_vertices()

    



# Visualize meshes
o3d.visualization.draw_geometries([fine_mesh], window_name="Fine Mesh")

n_triangles = np.asarray(simplified_mesh.triangles).shape[0]



o3d.visualization.draw_geometries([simplified_mesh], window_name="Simplified Mesh")




if manual_decimation == False:
    patches_name = shape_name + '_'+str(n_triangles)+'patch'
else:
    patches_name = shape_name + '_m'+'patch'


os.makedirs('patches/'+patches_name, exist_ok=True)
o3d.io.write_triangle_mesh('patches/'+patches_name+'/fine.obj', fine_mesh)

V_fine = np.asarray(fine_mesh.vertices)
V_simp = np.asarray(simplified_mesh.vertices)

# Snap the simplified mesh vertices to the fine mesh vertices
coarse_vertex_indices = cKDTree(V_fine).query(V_simp)[1]
V_coarse = V_fine[coarse_vertex_indices]

# Create new mesh with snapped vertices
coarse_mesh = o3d.geometry.TriangleMesh(
    vertices=o3d.utility.Vector3dVector(V_coarse),
    triangles=simplified_mesh.triangles
)
coarse_mesh.compute_vertex_normals()

o3d.visualization.draw_geometries([coarse_mesh], window_name="Coarse Mesh")

o3d.io.write_triangle_mesh("patches/"+patches_name+"/coarse.obj", coarse_mesh)


[Open3D WARNING] Write OBJ can not include triangle normals.


True

In [7]:
coarse_mesh=simplified_mesh

In [8]:
# Make halfedge meshes, including a halfedge index for each vertex.
coarse_he_mesh, coarse_V_he = he_mesh_from_o3d(coarse_mesh)
fine_he_mesh, fine_V_he = he_mesh_from_o3d(fine_mesh)

Preparing V_he for halfedge mesh
Prepared V_he for halfedge mesh.

Preparing V_he for halfedge mesh
10000 of 54846
20000 of 54846
30000 of 54846
40000 of 54846
50000 of 54846
Prepared V_he for halfedge mesh.



In [9]:
'''Find geodesics corresponding to coarse mesh vertices, and cut the fine mesh along the geodesics. '''

geodesic_vtx_indices = dict()
geodesic_pcds = []
half_edges = []

count = 1
total=len(coarse_he_mesh.half_edges)//2

#os.system('cp data/surfaces/'+shape_name+'.off ../geodesic_matlab/data/'+shape_name+'.off' )
#convert_off_file('../geodesic_matlab/data/'+shape_name+'.off', '../geodesic_matlab/data/'+shape_name+'.txt')


bdry_halfedge_mask = -1 * np.ones(len(fine_he_mesh.half_edges), dtype=int)



for i, he in enumerate(coarse_he_mesh.half_edges):

    v_id_start = coarse_vertex_indices [he.vertex_indices[0]]
    v_id_end = coarse_vertex_indices [he.vertex_indices[1]]

    left_face_id = he.triangle_index
    right_face_id = coarse_he_mesh.half_edges[he.twin].triangle_index

    if not ((v_id_start, v_id_end) in half_edges): # don't cut a halfedge if we already cut its twin
        
        print('Cutting the mesh along curve',str(count),'of',str(total)+'.')
        half_edges += [(v_id_start, v_id_end), (v_id_end, v_id_start)]
    
        print('start:', v_id_start, 'end:', v_id_end)


        # prepare .txt file
        fine_mesh = o3d_mesh_from_he_mesh(fine_he_mesh)
        o3d.io.write_triangle_mesh('../geodesic_matlab/data/temp.off', fine_mesh)
        o3d.io.write_triangle_mesh("patches/"+patches_name+"/fine.obj", fine_mesh)
        
        convert_off_file('../geodesic_matlab/data/temp.off', '../geodesic_matlab/data/temp.txt')
    
        # Call geodesic_matlab find_path
        os.chdir(original_cwd)
        os.chdir('../geodesic_matlab/src')  # or wherever `geodesic_matlab` is accessible from
        #os.system('./find_path_exact.out ../data/temp.txt '+str(v_id_end)+' '+str(v_id_start))
        if subdivision_level == 'infinity':
            os.system('./find_path_exact.out ../data/temp.txt '+str(v_id_end)+' '+str(v_id_start) )
        else:
            os.system('./find_path_subdivision.out ../data/temp.txt '+str(v_id_end)+' '+str(v_id_start) + ' ' + str(subdivision_level))
        os.chdir(original_cwd)
        
        geodesic_filepath = '../geodesic_matlab/paths/path.obj'
        os.system('cp ../geodesic_matlab/paths/path.obj patches/'+patches_name+'/path_'+str(i)+'.obj')
    
        fine_he_mesh, fine_V_he, geodesic_pcd, vtx_indices, bdry_halfedge_mask = cut_mesh_along_geodesic_v3(fine_he_mesh, fine_V_he,
                                                                                geodesic_filepath,
                                                                            bdry_halfedge_mask, v_id_start, v_id_end,
                                                                            left_face_id=left_face_id, 
                                                                            right_face_id=right_face_id, debug_plot=False )

        geodesic_vtx_indices[i] = vtx_indices
        geodesic_vtx_indices[he.twin] = vtx_indices[::-1]
    
        geodesic_pcds.append(geodesic_pcd)
        print(vtx_indices)

        #o3d.visualization.draw_geometries([he_mesh, geodesic_pcd], window_name="Cut along a curve successfully.")
    
        #o3d.visualization.draw_geometries([fine_he_mesh]+geodesic_pcds, window_name="Mesh Cutting Progress "+str(count)+" of "+str(total)+".")
    
        count+=1
        
fine_mesh = o3d_mesh_from_he_mesh(fine_he_mesh)
o3d.io.write_triangle_mesh("patches/"+patches_name+"/fine.obj", fine_mesh)

#o3d.visualization.draw_geometries([fine_he_mesh]+geodesic_pcds, window_name="Final Cut Mesh")
V_fine = np.asarray(fine_he_mesh.vertices)

Cutting the mesh along curve 1 of 708.
start: 574 end: 634
Processed file saved as ../geodesic_matlab/data/temp.txt

mesh has 9143 vertices, 18282 faces, 27423 edges
0 edges are boundary edges
shortest/longest edges are 0.00208449/0.0179787 = 0.115942
enclosing XYZ box: X[-0.288584,0.288584] Y[-0.5,0.5] Z[-0.0852534,0.0852534]
approximate diameter of the mesh is 1.16713
min/max face angles are 10.7037/147.457 degrees

Path saved to ../paths/path.obj
Starting the cutting for a curve. First and last vertex id are 574 634
Cut along a curve successfully
[2808, 634]
Cutting the mesh along curve 2 of 708.
start: 634 end: 5418
Processed file saved as ../geodesic_matlab/data/temp.txt

mesh has 9143 vertices, 18282 faces, 27423 edges
0 edges are boundary edges
shortest/longest edges are 0.00208449/0.0179787 = 0.115942
enclosing XYZ box: X[-0.288584,0.288584] Y[-0.5,0.5] Z[-0.0852534,0.0852534]
approximate diameter of the mesh is 1.16713
min/max face angles are 10.7037/147.457 degrees

Path save

UnboundLocalError: local variable 'vertex_indices' referenced before assignment

In [ ]:
'''
# Debugging vertex indices
half_edges = coarse_he_mesh.half_edges
i=5

g1 = geodesic_vtx_indices[i]
g2 = geodesic_vtx_indices[half_edges[i].next]
g3 = geodesic_vtx_indices[half_edges[half_edges[i].next].next]

plt.plot(V_fine[g1][:,0], V_fine[g1][:,1], alpha=0.2)
plt.plot(V_fine[g2][:,0], V_fine[g2][:,1], alpha=0.2)
plt.plot(V_fine[g3][:,0], V_fine[g3][:,1], alpha=0.2)
plt.show()

print(g1)
print(g2)
print(g3)
'''


In [ ]:
coarse_half_edges = coarse_he_mesh.half_edges
fine_half_edges = fine_he_mesh.half_edges

V = np.asarray(fine_he_mesh.vertices)
F = np.asarray(fine_he_mesh.triangles)

print(len(fine_half_edges))
print(bdry_halfedge_mask.shape)

# Step 3: Create a binary mask (example: highlight half of them)

#highlight_mask = (bdry_halfedge_mask>=0)

highlight_mask = (bdry_halfedge_mask>=0)

# Step 4: Create geometry to visualize highlighted halfedges
highlighted_lines = []
colors = []
line_points = []

for he_id in range(len(fine_half_edges)):
    
    if highlight_mask[he_id]:
       
        idx = len(line_points)
        
        start_id, end_id = fine_half_edges[he_id].vertex_indices
        #print(start_id,end_id)
        
        line_points.extend([V[start_id,:], V[end_id,:]])
        highlighted_lines.append([idx, idx + 1])
        colors.append([1, 0, 0])  # red lines

# Create line set for highlighted halfedges
line_set = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(line_points),
    lines=o3d.utility.Vector2iVector(highlighted_lines),
)
line_set.colors = o3d.utility.Vector3dVector(colors)

# Step 5: Visualize mesh with highlighted halfedges
o3d.visualization.draw_geometries([ line_set, fine_mesh])
o3d.visualization.draw_geometries([ line_set])



In [ ]:
''' Do floodfill to find the patches, given their boundaries. '''


import mesh_processing
import importlib
importlib.reload(mesh_processing)
from mesh_processing import *


coarse_half_edges = coarse_he_mesh.half_edges
fine_half_edges = fine_he_mesh.half_edges

V = np.asarray(fine_he_mesh.vertices)
F = np.asarray(fine_he_mesh.triangles)

coarse_V = np.asarray(coarse_he_mesh.vertices)
coarse_F = np.asarray(coarse_he_mesh.triangles)

selected_faces = []

union_mask = np.array([False for i in range(len(F))])

patch_meshes = []


N = len(fine_half_edges)

bdry_mask_as_list = bdry_halfedge_mask.tolist()

for i in range(coarse_F.shape[0]):
    print(i)

    selected_faces=[]
    seed_he_index = 0
    faces_added=False

    while faces_added==False:
        seed_he_index = bdry_mask_as_list.index(i, seed_he_index)
        
        selected_faces = floodfill(seed_he_index, fine_half_edges, bdry_halfedge_mask, V)
    
        mask = np.array([i in selected_faces for i in range(len(F))]) #A is region inside
    
        
        
        F_part = F[mask]
    
        if np.any(   np.logical_and(mask, ~union_mask) ):
            faces_added=True
        union_mask = np.logical_or(union_mask, mask)
        faces_added=True
        

    # Create new mesh
    A = o3d.geometry.TriangleMesh()
    A.vertices = o3d.utility.Vector3dVector(V)
    A.triangles = o3d.utility.Vector3iVector(F_part)
    A.paint_uniform_color(np.random.random(3)) # assign a colour
    A.compute_vertex_normals()

    

    A.remove_unreferenced_vertices()
    patch_meshes.append(A)
    o3d.visualization.draw_geometries([A])
    o3d.io.write_triangle_mesh("patches/"+patches_name+"/"+str(i)+".obj", A)

'''
while not np.all(union_mask==True):

    seed_he_index = bdry_mask_as_list.index(i)
    
    selected_faces = floodfill(seed_he_index, fine_half_edges, bdry_halfedge_mask, V)

    mask = np.array([i in selected_faces for i in range(len(F))]) #A is region inside

    if np.any(   np.logical_and(mask, !union_mask) ):

        union_mask = np.logical_or(union_mask, mask)
        
        F_part = F[mask]
            
    
        # Create new mesh
        A = o3d.geometry.TriangleMesh()
        A.vertices = o3d.utility.Vector3dVector(V)
        A.triangles = o3d.utility.Vector3iVector(F_part)
        A.paint_uniform_color(np.random.random(3)) # assign a colour
        A.compute_vertex_normals()
    
        
    
        A.remove_unreferenced_vertices()
        patch_meshes.append(A)
        o3d.io.write_triangle_mesh("patches/"+patches_name+"/"+str(i)+".obj", A)

'''

    
    
        
o3d.visualization.draw_geometries(patch_meshes)
#for A in patch_meshes:
#    o3d.visualization.draw_geometries([A])

#####################################################



    

In [ ]:
''' Add some special extra data to the patch obj files (the indices that correspond to corners) '''

# Loop over each mesh in `patch_meshes`
for i in range(n_triangles):

    A = o3d.io.read_triangle_mesh("patches/"+patches_name+"/"+str(i)+".obj")
    

    # Get the 3 vertices (abc) from coarse mesh
    abc = coarse_V[coarse_F[i]]  # `abc` should be a (3, 3) array

    # Convert mesh A's vertices to numpy array
    V = np.asarray(A.vertices)
    F = np.asarray(A.triangles)

    # Build a cKDTree for fast nearest-neighbor search in mesh A's vertices
    kdtree = cKDTree(V)

    # Find closest indices in V for each vertex in abc
    dist, closest_indices = kdtree.query(abc)  # The result should be an array of shape (3,) with the indices

    print('dist', dist)
    # Create the comment with the special indices
    comment = f"# Coarse Triangle Correspondence: {closest_indices[0]}, {closest_indices[1]}, {closest_indices[2]}"

    # Save the mesh to a temporary .obj file using Open3D
    obj_file_path = f"patches/{patches_name}/{i}.obj"
    o3d.io.write_triangle_mesh(obj_file_path, A)

    # Open the file and insert the comment at the top
    with open(obj_file_path, "r") as file:
        lines = file.readlines()

    # Prepend the comment line to the file content
    lines.insert(0, comment + "\n")

    # Write the content back to the file
    with open(obj_file_path, "w") as file:
        file.writelines(lines)

    print(f"Triangle {i}: Closest indices in V for abc: {closest_indices}")
    print(f"Mesh saved at: {obj_file_path} with special indices as comments.")

In [ ]:
''' Finally, parametrise the triangles and save as {i}_param.obj '''


import importlib
import visuals
importlib.reload(visuals)

from visuals import *

import mesh_processing
importlib.reload(mesh_processing)

from mesh_processing import *


for i in range(n_triangles):
    input_filename = "patches/"+patches_name+"/"+str(i)+".obj"
    output_filename = "patches/"+patches_name+"/"+str(i)+"_param.obj"
    tutte_embedding(input_filename, output_filename, visualise=False)